# GEO PERU - DATOS COMPLETOS DE CENSO 2017



In [14]:

# Verifica qué hay en el archivo antes de procesarlo
import pandas as pd

# Carga solo las primeras 20 filas de la primera hoja
df_check = pd.read_excel("Downloads/01TOMO_01.xlsx", sheet_name=0, header=None)
print("Primeras 20 filas del archivo:")
print(df_check.head(20))

Primeras 20 filas del archivo:
                                                    0       1           2  \
0   CUADRO Nº 1: POBLACIÓN CENSADA, POR ÁREA URBAN...     NaN         NaN   
1               Provincia, distrito, y edades simples   Total  Población    
2                                                 NaN     NaN     Hombres   
3                                                 NaN     NaN         NaN   
4                               DEPARTAMENTO AMAZONAS  379384      191275   
5                                                 NaN     NaN         NaN   
6                                    Menores de 1 año    7098        3521   
7                                       De 1 a 4 años   31584       15968   
8                                               1 año    7436        3777   
9                                              2 años    7796        3876   
10                                             3 años    8047        4089   
11                                           

In [16]:
"""
Script para extraer datos del Censo 2017 INEI - Estructura jerárquica
Funciona para SALUD1 (seguro de salud por distrito/área/edad)
y EDUC (asistencia escolar, alfabetismo)

PROBLEMA DEL EXCEL INEI:
- La geografía (departamento, provincia, distrito) está como VALOR en col A
- Los números usan ESPACIO como separador de miles: "357 054"
- Hay filas de subtotales (Hombres, Mujeres, grupos de edad) mezcladas
- Las filas de lugar geográfico tienen el nombre en MAYÚSCULAS y en negrita
"""

import pandas as pd
import re
import os


def limpiar_numero(val):
    """Convierte '357 054' o '7 208 221' o '-' a entero."""
    if pd.isna(val):
        return 0
    s = str(val).strip()
    if s in ['-', '', 'nan']:
        return 0
    # Eliminar espacios (separador de miles del INEI)
    s = s.replace(' ', '').replace(',', '')
    try:
        return int(float(s))
    except:
        return 0


def es_fila_geografica(texto):
    """
    Detecta si una fila es un encabezado geográfico (departamento/provincia/distrito).
    En los Excel del INEI, estos aparecen en MAYÚSCULAS o con patrones específicos.
    Ejemplos: 'DEPARTAMENTO AMAZONAS', 'PROVINCIA CHACHAPOYAS', 'DISTRITO CHACHAPOYAS'
    """
    if pd.isna(texto):
        return False
    t = str(texto).strip().upper()
    # Encabezados geográficos del INEI
    prefijos = ['DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'REGION', 'REGIÓN']
    for p in prefijos:
        if t.startswith(p):
            return True
    return False


def es_fila_a_excluir(texto):
    """
    Filas que NO son geográficas y deben ignorarse:
    subtotales de sexo, grupos de edad, nivel educativo, área, etc.
    """
    if pd.isna(texto):
        return True
    t = str(texto).strip()
    excluir = [
        'Hombres', 'Mujeres', 'Total',
        'Urbana', 'Rural',
        'Sin nivel', 'Inicial', 'Primaria', 'Secundaria',
        'Básica especial',
        'Sup. no univ. incompleta', 'Sup. no univ. completa',
        'Sup. univ. incompleta', 'Sup. univ. completa',
        'Maestría / Doctorado',
        'Menores de 1 año', 'De 1 a 14 años', 'De 15 a 29 años',
        'De 30 a 44 años', 'De 45 a 64 años', 'De 65 y más años',
        'Sin dificultad', 'Con dificultad',
        'Quechua', 'Aymara', 'Castellano',
        'No asiste', 'Asiste',
    ]
    for e in excluir:
        if t == e or t.startswith(e):
            return True
    # Excluir si es solo números o vacío
    if re.match(r'^[\d\s\.,\-]+$', t):
        return True
    return False


def extraer_salud1(ruta_excel, nombre_departamento):
    """
    Extrae del CUADRO 1 (SALUD1): afiliación a seguro de salud
    por provincia, distrito, área urbana/rural.
    
    Retorna DataFrame con columnas:
    departamento, nivel_geo, nombre_lugar, area,
    total, SIS, ESSALUD, FFAA, privado, otro, sin_seguro, pct_sin_seguro
    """
    print(f"\n{'='*50}")
    print(f"Procesando: {nombre_departamento}")
    print(f"Archivo: {ruta_excel}")
    
    # --- PASO 1: Leer el Excel sin asumir estructura ---
    try:
        df_raw = pd.read_excel(
            ruta_excel,
            sheet_name="SALUD1",
            header=None,
            skiprows=4,      # Saltar título del cuadro (filas 1-4 aprox)
        )
    except Exception as e:
        print(f"ERROR al leer SALUD1: {e}")
        print("Intentando con skiprows=5...")
        df_raw = pd.read_excel(
            ruta_excel,
            sheet_name="SALUD1", 
            header=None,
            skiprows=5,
        )
    
    print(f"Filas leídas: {len(df_raw)}")
    print(f"Columnas: {df_raw.shape[1]}")
    print(f"\nPrimeras 10 filas columna A:")
    print(df_raw.iloc[:10, 0].tolist())
    
    # --- PASO 2: Identificar columnas ---
    # El INEI siempre tiene: col A = geo, B = Total, C = SIS, D = ESSALUD,
    # E = FFAA, F = privado, G = otro, H = ninguno
    # Pero a veces hay columnas vacías extras
    
    # Encontrar la columna con números grandes (total de población)
    # buscando la primera fila con dato numérico grande
    col_nombres = {
        0: 'geo',
        1: 'total',
        2: 'SIS',
        3: 'ESSALUD',
        4: 'FFAA',
        5: 'privado',
        6: 'otro',
        7: 'sin_seguro'
    }
    
    # Renombrar columnas disponibles
    n_cols = df_raw.shape[1]
    nuevos_nombres = []
    for i in range(n_cols):
        nuevos_nombres.append(col_nombres.get(i, f'col_{i}'))
    df_raw.columns = nuevos_nombres
    
    # --- PASO 3: Rastrear contexto geográfico y extraer filas útiles ---
    resultados = []
    
    dept_actual = nombre_departamento
    provincia_actual = None
    distrito_actual = None
    area_actual = 'Total'
    
    for idx, row in df_raw.iterrows():
        geo_val = row.get('geo', None)
        
        if pd.isna(geo_val):
            continue
            
        geo_str = str(geo_val).strip()
        geo_upper = geo_str.upper()
        
        # Detectar nivel geográfico
        if 'DEPARTAMENTO' in geo_upper:
            dept_actual = geo_str
            continue
        elif 'PROVINCIA' in geo_upper:
            provincia_actual = geo_str.replace('PROVINCIA ', '').replace('Provincia ', '')
            distrito_actual = None
            continue
        elif 'DISTRITO' in geo_upper:
            distrito_actual = geo_str.replace('DISTRITO ', '').replace('Distrito ', '')
            area_actual = 'Total'
            continue
        
        # Detectar área urbana/rural
        if geo_str in ['Urbana', 'URBANA']:
            area_actual = 'Urbana'
            # Guardar fila de área
            if 'total' in row and not pd.isna(row['total']):
                fila = _construir_fila(
                    dept_actual, provincia_actual, distrito_actual,
                    area_actual, row
                )
                if fila:
                    resultados.append(fila)
            continue
        elif geo_str in ['Rural', 'RURAL']:
            area_actual = 'Rural'
            if 'total' in row and not pd.isna(row['total']):
                fila = _construir_fila(
                    dept_actual, provincia_actual, distrito_actual,
                    area_actual, row
                )
                if fila:
                    resultados.append(fila)
            continue
        
        # Fila de total del distrito (viene justo después del nombre del distrito)
        # Se identifica porque geo_str es el nombre del distrito pero sin "DISTRITO"
        # o porque el total tiene un número grande
        if not es_fila_a_excluir(geo_str) and not es_fila_geografica(geo_str):
            if 'total' in row:
                total_val = limpiar_numero(row.get('total', 0))
                if total_val > 0:
                    fila = _construir_fila(
                        dept_actual, provincia_actual, distrito_actual,
                        'Total', row
                    )
                    if fila:
                        resultados.append(fila)
    
    if not resultados:
        print("⚠️  No se encontraron filas. Revisando estructura...")
        _debug_estructura(df_raw)
        return pd.DataFrame()
    
    df_resultado = pd.DataFrame(resultados)
    
    # Calcular porcentaje
    df_resultado['pct_sin_seguro'] = (
        df_resultado['sin_seguro'] / df_resultado['total'].replace(0, 1) * 100
    ).round(2)
    
    print(f"\n✓ Extraídas {len(df_resultado)} filas")
    print(df_resultado.head(10).to_string())
    
    return df_resultado


def _construir_fila(dept, provincia, distrito, area, row):
    """Construye un diccionario con los valores limpios de una fila."""
    total = limpiar_numero(row.get('total', 0))
    if total == 0:
        return None
    
    return {
        'departamento': dept,
        'provincia': provincia,
        'distrito': distrito,
        'area': area,
        'total': total,
        'SIS': limpiar_numero(row.get('SIS', 0)),
        'ESSALUD': limpiar_numero(row.get('ESSALUD', 0)),
        'FFAA': limpiar_numero(row.get('FFAA', 0)),
        'privado': limpiar_numero(row.get('privado', 0)),
        'otro': limpiar_numero(row.get('otro', 0)),
        'sin_seguro': limpiar_numero(row.get('sin_seguro', 0)),
    }


def _debug_estructura(df_raw):
    """Muestra la estructura real del archivo para diagnosticar."""
    print("\n--- DEBUG: primeras 30 filas columna A ---")
    for i, val in enumerate(df_raw.iloc[:30, 0]):
        print(f"  Fila {i:3d}: '{val}'")
    print("\n--- DEBUG: valores únicos columna A (primeros 50) ---")
    unicos = df_raw.iloc[:, 0].dropna().unique()[:50]
    for u in unicos:
        print(f"  '{u}'")


# ============================================================
# VERSIÓN SIMPLIFICADA — más robusta para estructura INEI real
# ============================================================

def extraer_salud_simple(ruta_excel, sheet='SALUD1', skiprows=4):
    """
    Versión simplificada: lee TODO el Excel y filtra las filas
    que corresponden a DISTRITOS (en mayúsculas con 'DISTRITO').
    
    Esta es la más robusta porque no depende de rastrear contexto.
    Solo extrae los totales por distrito.
    """
    print(f"\nLeyendo {sheet} de {ruta_excel}...")
    
    df = pd.read_excel(
        ruta_excel,
        sheet_name=sheet,
        header=None,
        skiprows=skiprows,
        dtype=str  # ← CLAVE: leer todo como texto primero
    )
    
    # Renombrar columnas
    col_map = {0:'geo', 1:'total', 2:'SIS', 3:'ESSALUD',
               4:'FFAA', 5:'privado', 6:'otro', 7:'sin_seguro'}
    df.rename(columns={i: col_map.get(i, f'c{i}') 
                        for i in range(df.shape[1])}, inplace=True)
    
    print(f"Shape: {df.shape}")
    print(f"Primeras filas col A: {df['geo'].dropna().head(15).tolist()}")
    
    # Limpiar la columna geo
    df['geo'] = df['geo'].astype(str).str.strip()
    
    # Filtrar filas de DISTRITO (las que tienen el total del distrito)
    mask_distrito = df['geo'].str.upper().str.startswith('DISTRITO')
    df_distritos = df[mask_distrito].copy()
    
    print(f"\nFilas de DISTRITO encontradas: {len(df_distritos)}")
    
    if len(df_distritos) == 0:
        print("⚠️  No hay filas con 'DISTRITO'. Mostrando valores únicos col A:")
        print(df['geo'].dropna().unique()[:30])
        return df  # Retornar todo para inspección
    
    # Limpiar números
    for col in ['total', 'SIS', 'ESSALUD', 'FFAA', 'privado', 'otro', 'sin_seguro']:
        if col in df_distritos.columns:
            df_distritos[col] = df_distritos[col].apply(limpiar_numero)
    
    # Nombre limpio del distrito
    df_distritos['nombre_distrito'] = (
        df_distritos['geo']
        .str.replace('DISTRITO ', '', case=False)
        .str.strip()
        .str.title()
    )
    
    # Calcular porcentaje
    df_distritos['pct_sin_seguro'] = (
        df_distritos['sin_seguro'] / 
        df_distritos['total'].replace(0, 1) * 100
    ).round(2)
    
    cols_final = ['nombre_distrito', 'total', 'SIS', 'ESSALUD', 
                  'sin_seguro', 'pct_sin_seguro']
    resultado = df_distritos[[c for c in cols_final if c in df_distritos.columns]]
    
    print(f"\n✓ Resultado:")
    print(resultado.to_string(index=False))
    
    return resultado


# ============================================================
# USO PRINCIPAL
# ============================================================

if __name__ == "__main__":
    
    # AJUSTA ESTA RUTA a tu archivo
    ruta = "Downloads/01TOMO_03.xlsx"  # o la ruta completa
    
    if not os.path.exists(ruta):
        print(f"Archivo no encontrado: {ruta}")
        print("Ajusta la variable 'ruta' con la ruta correcta")
        print("\nEjemplo de uso:")
        print("  df = extraer_salud_simple('Downloads/01TOMO_03.xlsx', sheet='SALUD5')")
    else:
        # Primero probar la versión simple (más robusta)
        df_resultado = extraer_salud_simple(ruta, sheet='SALUD5')
        
        if len(df_resultado) > 0:
            # Guardar resultado
            output = ruta.replace('.xlsx', '_salud_limpio.csv')
            df_resultado.to_csv(output, index=False, encoding='utf-8')
            print(f"\n✓ Guardado en: {output}")


Leyendo SALUD5 de Downloads/01TOMO_03.xlsx...
Shape: (7799, 8)
Primeras filas col A: ['DEPARTAMENTO AMAZONAS', 'Sin nivel', 'Inicial', 'Primaria', 'Secundaria', 'Básica especial', 'Sup. no univ. incompleta', 'Sup. no univ. completa', 'Sup. univ. incompleta', 'Sup. univ. completa', 'Maestría / Doctorado', 'Hombres', 'Sin nivel', 'Inicial', 'Primaria']

Filas de DISTRITO encontradas: 84

✓ Resultado:
        nombre_distrito  total   SIS  ESSALUD  sin_seguro  pct_sin_seguro
            Chachapoyas  31036 14195     9907        5830           18.78
               Asunción    250   207       28          12            4.80
                 Balsas   1066   761       92         200           18.76
                  Cheto    612   512       26          70           11.44
              Chiliquín    557   498       22          31            5.57
            Chuquibamba   1683  1371      132         165            9.80
                Granada    455   370       14          65           14.29
     

In [9]:
import pandas as pd
from IPython.display import FileLink

def extraer_con_ubigeo(ruta_archivo):
    # Leemos el archivo
    df = pd.read_excel(ruta_archivo, header=None)
    
    # 1. Filtramos las filas rojas que contienen "DISTRITO" en la columna 1
    df_rojas = df[df[1].astype(str).str.contains('DISTRITO', na=False)].copy()
    
    # 2. Seleccionamos: 
    # Columna 0: Código (Ubigeo)
    # Columna 1: Nombre (Distrito)
    # Columna 6: Hombres
    # Columna 7: Mujeres
    # Columna 8: Total_Viviendas
    df_rojas = df_rojas[[0, 1, 6, 7, 8]]
    df_rojas.columns = ['Ubigeo', 'Distrito', 'Hombres', 'Mujeres', 'Total_Viviendas']
    
    # 3. Limpieza de números
    for col in ['Hombres', 'Mujeres', 'Total_Viviendas']:
        df_rojas[col] = df_rojas[col].astype(str).str.replace(r'[\s,]', '', regex=True)
        df_rojas[col] = pd.to_numeric(df_rojas[col].replace('-', '0'), errors='coerce').fillna(0).astype(int)
    
    return df_rojas

# --- EJECUCIÓN ---
ruta = "Downloads/DataGeneralDeAmazonas.xlsx" 
df_final = extraer_con_ubigeo(ruta)

# Guardar con separador de punto y coma
nombre_csv = "Distritos_con_Ubigeo.csv"
df_final.to_csv(nombre_csv, index=False, encoding='utf-8-sig', sep=';')

print("✅ Archivo generado con éxito incluyendo el código de Ubigeo.")
display(FileLink(nombre_csv))

✅ Archivo generado con éxito incluyendo el código de Ubigeo.


C:\Users\Usuario\Distritos_con_Ubigeo.csv

In [3]:
import pandas as pd
import unicodedata
import os
import glob

def normalizar_texto(texto):
    """Elimina tildes, espacios extra y pasa a mayúsculas para asegurar el JOIN con UBIGEO."""
    if pd.isna(texto):
        return ""
    texto = str(texto).strip().upper()
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto)
                  if unicodedata.category(c) != 'Mn')
    return texto

def extraer_doble_vulnerabilidad(ruta_excel, sheet='SALUD5', skiprows=4):
    print(f"\nProcesando: {os.path.basename(ruta_excel)}")
    
    # Leer como texto para evitar pérdida de datos
    df = pd.read_excel(ruta_excel, sheet_name=sheet, header=None, skiprows=skiprows, dtype=str)
    
    # 1. Mapeo de columnas (Alineado a SALUD5)
    col_map = {0:'geo', 1:'total_segmento', 2:'SIS', 3:'ESSALUD', 4:'FFAA', 5:'privado', 6:'otro', 7:'sin_seguro'}
    df.rename(columns=col_map, inplace=True)
    df = df[['geo', 'total_segmento', 'SIS', 'ESSALUD', 'FFAA', 'privado', 'otro', 'sin_seguro']].copy()
    
    df['geo'] = df['geo'].astype(str).str.strip()

    # 2. RASTREO DE JERARQUÍA PROFUNDO
    # Ahora también arrastramos el distrito hacia abajo para asociarlo a sus sub-filas
    df['Departamento'] = df['geo'].where(df['geo'].str.upper().str.startswith('DEPARTAMENTO')).ffill()
    df['Provincia'] = df['geo'].where(df['geo'].str.upper().str.startswith('PROVINCIA')).ffill()
    df['Distrito'] = df['geo'].where(df['geo'].str.upper().str.startswith('DISTRITO')).ffill()
    
    # Eliminar filas basura de arriba (donde Distrito aún no existe)
    df = df.dropna(subset=['Distrito'])

    # 3. Filtrar solo la fila de interés ("Sin nivel" educativo)
    mask_sin_nivel = df['geo'] == 'Sin nivel'
    df_vuln = df[mask_sin_nivel].copy()
    
    if df_vuln.empty:
        return pd.DataFrame()

    # 4. Quedarnos solo con el Total por distrito (evitar duplicados de Hombres/Mujeres)
    # Al mantener el 'first', aseguramos que toma el bloque principal antes del desglose por sexo
    df_vuln = df_vuln.drop_duplicates(subset=['Departamento', 'Provincia', 'Distrito'], keep='first')

    # Limpiar prefijos y normalizar
    df_vuln['Departamento'] = df_vuln['Departamento'].str.replace('DEPARTAMENTO ', '', case=False).apply(normalizar_texto)
    df_vuln['Provincia'] = df_vuln['Provincia'].str.replace('PROVINCIA ', '', case=False).apply(normalizar_texto)
    df_vuln['Distrito'] = df_vuln['Distrito'].str.replace('DISTRITO ', '', case=False).apply(normalizar_texto)

    # 5. Limpieza de números vectorizada
    cols_numericas = ['total_segmento', 'SIS', 'ESSALUD', 'FFAA', 'privado', 'otro', 'sin_seguro']
    for col in cols_numericas:
        df_vuln[col] = pd.to_numeric(
            df_vuln[col].astype(str).str.replace(r'[\s,]', '', regex=True).str.replace('-', '0'), 
            errors='coerce'
        ).fillna(0).astype(int)

    # 6. Cálculo del indicador de Doble Vulnerabilidad
    # Calcula qué porcentaje de los que "no tienen nivel educativo" tampoco tienen seguro
    df_vuln['pct_sin_seguro_en_sin_nivel'] = (df_vuln['sin_seguro'] / df_vuln['total_segmento'].replace(0, 1) * 100).round(2)

    # 7. Renombrar columnas finales para que el CSV sea autodescriptivo
    cols_finales = ['Departamento', 'Provincia', 'Distrito', 'total_segmento', 'sin_seguro', 'pct_sin_seguro_en_sin_nivel']
    df_final = df_vuln[cols_finales].rename(columns={
        'total_segmento': 'pob_sin_nivel_educativo',
        'sin_seguro': 'sin_nivel_y_sin_seguro'
    })
    
    return df_final

# Uso para procesar UN SOLO ARCHIVO de prueba
if __name__ == "__main__":
    
    # Ajusta tu ruta aquí
    ruta_archivo = "Downloads/01TOMO_03.xlsx" 
    
    if not os.path.exists(ruta_archivo):
        print(f"❌ ERROR: Python no encuentra el archivo en la ruta: {ruta_archivo}")
    else:
        print("✅ Archivo encontrado. Iniciando extracción de Doble Vulnerabilidad...")
        
        # Procesamos la hoja SALUD5
        df_limpio = extraer_doble_vulnerabilidad(ruta_archivo, sheet='SALUD5')
        
        if not df_limpio.empty:
            ruta_salida = "Downloads/Atlas_DobleVuln_Amazonas.csv"
            df_limpio.to_csv(ruta_salida, index=False, encoding='utf-8')
            print(f"\n✓ Procesamiento completado.")
            print(f"✓ Total de distritos extraídos: {len(df_limpio)}")
            print(f"✓ Guardado en: {ruta_salida}")
            
            # Mostrar una vista previa en la terminal para confirmar
            print("\nVista previa de los primeros 3 registros:")
            print(df_limpio.head(3).to_string(index=False))
        else:
            print("\n⚠️ El script corrió, pero no extrajo datos.")

✅ Archivo encontrado. Iniciando extracción de Doble Vulnerabilidad...

Procesando: 01TOMO_03.xlsx

✓ Procesamiento completado.
✓ Total de distritos extraídos: 90
✓ Guardado en: Downloads/Atlas_DobleVuln_Amazonas.csv

Vista previa de los primeros 3 registros:
Departamento   Provincia    Distrito  pob_sin_nivel_educativo  sin_nivel_y_sin_seguro  pct_sin_seguro_en_sin_nivel
    AMAZONAS CHACHAPOYAS CHACHAPOYAS                     1231                     198                        16.08
    AMAZONAS CHACHAPOYAS    ASUNCION                       38                       1                         2.63
    AMAZONAS CHACHAPOYAS      BALSAS                      123                      26                        21.14
